# Synthetic Trade Generator

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

class SyntheticTradeGenerator:
    """
    Generates simulated Front Office (FO) and Back Office (BO) trade feeds.
    """
    
    def __init__(self, num_trades=100, random_state=42):
        self.num_trades = num_trades
        self.random_state = random_state
        
        # Lock in the random seed for reproducible datasets
        np.random.seed(self.random_state)
        
        self.product_types = ['Equity Index Option', 'Total Return Swap', 'Stock Loan']
        self.counterparties = ['VectaLink Global Ltd.', 'Citadel', 'BlackRock', 'Millennium']

    def _generate_base_trades(self):
        """Generates the clean Front Office dataset."""
        # Vectorized generation for speed and clean syntax
        days_ago = np.random.randint(0, 10, self.num_trades)
        trade_dates = [datetime.now() - timedelta(days=int(d)) for d in days_ago]
        
        data = {
            'Trade_ID': [f"TRD-{1000 + i}" for i in range(self.num_trades)],
            'Product_Type': np.random.choice(self.product_types, self.num_trades),
            'Counterparty': np.random.choice(self.counterparties, self.num_trades),
            'Notional_GBP': np.round(np.random.uniform(500000, 5000000, self.num_trades), 2),
            'Strike_Price': np.round(np.random.uniform(100, 500, self.num_trades), 2),
            'MTM_GBP': np.round(np.random.uniform(-100000, 100000, self.num_trades), 2),
            'Day_Count_Basis': np.where(np.random.rand(self.num_trades) > 0.5, 'Actual/365', '30/360'),
            'Trade_Date': [d.strftime('%Y-%m-%d') for d in trade_dates],
            'Status': ['Confirmed'] * self.num_trades
        }
        
        return pd.DataFrame(data)

    def generate_feeds(self):
        # Create pure FO feed
        fo_df = self._generate_base_trades()
        
        # Clone FO feed to create BO feed, then inject errors
        bo_df = fo_df.copy()
        
        # Select 15 unique random indices for error injection
        break_indices = np.random.choice(self.num_trades, size=15, replace=False)
        
        # Break 1: Accrual Mismatch
        for idx in break_indices[0:5]:
            current_basis = fo_df.at[idx, 'Day_Count_Basis']
            bo_df.at[idx, 'Day_Count_Basis'] = '30/360' if current_basis == 'Actual/365' else 'Actual/365'
            
        # Break 2: MTM Discrepancy 
        for idx in break_indices[5:10]:
            error_margin = np.random.uniform(50000, 150000)
            bo_df.at[idx, 'MTM_GBP'] = fo_df.at[idx, 'MTM_GBP'] + error_margin
            
        # Break 3: Unconfirmed Status 
        for idx in break_indices[10:15]:
            bo_df.at[idx, 'Status'] = 'Unconfirmed'
            
        # Break 4: Missing Trades in BO (Drop 2 random rows entirely)
        drop_indices = np.random.choice(self.num_trades, size=2, replace=False)
        bo_df = bo_df.drop(drop_indices)
        
        return fo_df, bo_df

In [2]:
# Instantiate the class and generate the data
generator = SyntheticTradeGenerator(num_trades=100)
fo_feed, bo_feed = generator.generate_feeds()

# Check the shapes (BO should have 98 rows due to the 2 dropped captures)
print(f"FO Feed Shape: {fo_feed.shape}")
print(f"BO Feed Shape: {bo_feed.shape}")

# Export for Module 2
fo_feed.to_csv('fo_feed.csv', index=False)
bo_feed.to_csv('bo_feed.csv', index=False)

FO Feed Shape: (100, 9)
BO Feed Shape: (98, 9)


In [3]:
class TradeReconciler:
    """
    Reconciles Front Office (FO) and Back Office (BO) trade feeds.
    """
    
    def __init__(self, fo_feed, bo_feed):
        self.fo_df = fo_feed
        self.bo_df = bo_feed

    def _join_feeds(self):
        """Joins FO and BO feeds on Trade_ID for reconciliation."""
        return pd.merge(self.fo_df, self.bo_df, on='Trade_ID', how='outer', suffixes=('_FO', '_BO'))

    def detect_breaks(self):
        # Detects discrepancies between FO and BO feeds.
        nerged_df = self._join_feeds()
        breaks = []

        for index, row in merged_df.interrows():
            if pd.isna(row['MTM_GBP_FO']) or pd.isna(row['MTM_GBP_BO']):
                breaks.append((row['Trade_ID'], 'Missing Trade'))